# Distributed Training with Ray Train + PyTorch
**Task:** Classifying wine cultivars (3 classes, 13 features) using an MLP trained across 2 Ray workers.


In [1]:
!pip install -q "ray[train]" torch scikit-learn pandas


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 1. Dataset

In [2]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

wine = load_wine()
X, y = wine.data.astype(np.float32), wine.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {wine.target_names}")

Train: (142, 13), Test: (36, 13)
Classes: ['class_0' 'class_1' 'class_2']


## 2. Model

In [3]:
import torch
import torch.nn as nn

class WineMLP(nn.Module):
    def __init__(self, input_dim=13, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

## 3. Putting data in Ray object store

In [4]:
import ray

ray.init(num_cpus=2, ignore_reinit_error=True)

# Stored once in shared memory — all workers read from here
X_train_ref = ray.put(X_train)
y_train_ref = ray.put(y_train)

print("Data in object store:", X_train_ref, y_train_ref)

2026-03-27 14:20:59,246	INFO worker.py:2013 -- Started a local Ray instance.


Data in object store: ObjectRef(00ffffffffffffffffffffffffffffffffffffff0100000001e1f505) ObjectRef(00ffffffffffffffffffffffffffffffffffffff0100000002e1f505)


/home/aparup/Workspace/Python/MLOps-Labs/labs/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## 4. Per-worker training function

In [5]:
import os
import tempfile
import ray.train as train
import ray.train.torch
from ray.train import Checkpoint
from torch.utils.data import DataLoader, TensorDataset, DistributedSampler

def train_func(config):
    lr         = config["lr"]
    batch_size = config["batch_size"]
    epochs     = config["epochs"]

    # Retrieve data from object store
    X = ray.get(config["X_ref"])
    y = ray.get(config["y_ref"])

    # DistributedSampler splits rows across workers — no sample is seen twice per epoch
    dataset = TensorDataset(torch.tensor(X), torch.tensor(y))
    sampler = DistributedSampler(dataset)
    loader  = DataLoader(dataset, batch_size=batch_size, sampler=sampler)

    # prepare_model() wraps in DistributedDataParallel
    model     = ray.train.torch.prepare_model(WineMLP())
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        total_loss, correct, total = 0.0, 0, 0

        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss    = criterion(outputs, y_batch)
            loss.backward()   # gradients averaged across workers automatically
            optimizer.step()

            total_loss += loss.item()
            correct    += (outputs.argmax(1) == y_batch).sum().item()
            total      += y_batch.size(0)

        train.report({
            "epoch":    epoch + 1,
            "loss":     total_loss / len(loader),
            "accuracy": correct / total
        })

    # Save checkpoint after training — use .module to unwrap DistributedDataParallel
    with tempfile.TemporaryDirectory() as tmpdir:
        torch.save(model.module.state_dict(), os.path.join(tmpdir, "model.pt"))
        train.report({"epoch": epochs, "done": True},
                     checkpoint=Checkpoint.from_directory(tmpdir))

## 5. Launching distributed training

In [6]:
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig

trainer = TorchTrainer(
    train_loop_per_worker=train_func,
    train_loop_config={
        "lr":         1e-3,
        "batch_size": 16,
        "epochs":     20,
        "X_ref":      X_train_ref,
        "y_ref":      y_train_ref,
    },
    scaling_config=ScalingConfig(num_workers=2, use_gpu=False),
)

result = trainer.fit()
print("Training complete.")

(TrainController pid=15803) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2


(RayTrainWorker pid=15970) [Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


(RayTrainWorker pid=15971) Setting up process group for: env:// [rank=0, world_size=2]
(PlacementGroupCleaner pid=15864) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(TrainController pid=15803) Started training worker group of size 2: 
(TrainController pid=15803) - (ip=10.0.0.201, pid=15971) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=15803) - (ip=10.0.0.201, pid=15970) world_rank=1, local_rank=1, node_rank=0
(RayTrainWorker pid=15971) Moving model to device: cpu
(RayTrainWorker pid=15971) Wrapping provided model in DistributedDataParallel.
(PlacementGroupCleaner pid=15864) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(RayTrainWorker pid=15971) Reporting training result 1: TrainingReport(checkpoint=None, metrics={'epoch': 1, 'loss': 1.112531805038452, 'accuracy': 0.29577464788732394}, validation=False)
(RayTrainWorker pid=15971) Re

Training complete.


(PlacementGroupCleaner pid=15864) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


## 6. Evaluating on test set

In [7]:
checkpoint = result.checkpoint

with checkpoint.as_directory() as tmpdir:
    state_dict = torch.load(os.path.join(tmpdir, "model.pt"))
    # Strip "module." prefix added by DistributedDataParallel
    state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
    model = WineMLP()
    model.load_state_dict(state_dict)

model.eval()
with torch.no_grad():
    preds = model(torch.tensor(X_test)).argmax(1).numpy()

print(f"Test accuracy: {(preds == y_test).mean():.4f}\n")
for i, name in enumerate(wine.target_names):
    mask = y_test == i
    print(f"  {name}: {(preds[mask] == y_test[mask]).mean():.4f}")

Test accuracy: 0.9444

  class_0: 1.0000
  class_1: 1.0000
  class_2: 0.8000
